# Упрощение текста при помощи T5

Задание: дообучить модель (модификацию T5) для упрощения русскоязычных текстов.

Данные: отрывок из корпуса RuAdapt.

In [ ]:
! pip install transformers[torch]

В этом туториале мы будем по минимуму пользоваться функционалом библиотеки Transformers и обучим модель практически так же, как сделали бы это в pytorch.

Для начала загрузим данные.

In [ ]:
import pandas as pd

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/ML_training_data/ruadapt_enc.csv')

In [ ]:
train_df, val_df = train_test_split(df)

In [ ]:
train_df

,source,target
3504,Патриарх Московский и Всея Руси является морал...,Патриарх Московский и Всея Руси является морал...
7757,"Русь), в суровых климатических условиях шуба б...",Их носили богатые бояре и дворяне
4106,"Найти Кощея ему помогают животные, которых он ...","Найти Кощея ему помогают животные, которых он ..."
4567,"Представляет собой белое полотнище, пересечённ...","Представляет собой белое полотнище, пересечённ..."
7498,С ХVIII в. в России на Рождество было принято ...,С ХVIII в. на Рождество стали украшать ёлку.
...,...,...
827,"Их художественная ценность была невелика, мног...","Их художественная ценность была невелика, мног..."
1758,"С течением времени на месте бывших валов, рвов...","Со временем на месте бывших валов, рвов и стен..."
5807,"Массовому читателю Тёркин известен, в основном...","Массовому читателю Тёркин известен, в основном..."
6637,В последнее десятилетие жизни писателем были с...,В последнее десятилетие жизни писателем были с...


In [ ]:
source_train = [i.strip() for i in train_df['source'].tolist()]
target_train = [i.strip() for i in train_df['target'].tolist()]

In [ ]:
lines = list(zip(source_train, target_train))

In [ ]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

Backbone model - это та модель, которую мы дообучаем. Мы возьмем дистиллированную версию модели T5 (в ней остались только русские и немного английских векторов), предобученную для перефразирования, и попробуем обучить ее еще и упрощать тексты.

In [ ]:
backbone_model = 'cointegrated/rut5-base-paraphraser'
model = T5ForConditionalGeneration.from_pretrained(backbone_model)
tokenizer = T5Tokenizer.from_pretrained(backbone_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/977M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Посылаем модель на gpu и определяем стратегию оптимизации.

In [ ]:
model.cuda();
optimizer = torch.optim.Adam(params=[p for p in model.parameters() if p.requires_grad], lr=1e-5)

Мы будем обучаться следующим образом:


1.   Брать одну случайную пару "сложное предложение, простое предложение" из датасета;
2.   Токенизировать ее предобученным токенайзером модели;
3.   Делать forward и backward pass;
4.   То же самое повторим еще iterations раз.



In [ ]:
import random

In [ ]:
def get_batch(sentence):
    # наши батчи - это просто пары предложений
    return sentence[0], sentence[1]

Посмотрим, что происходит при токенизации:

In [ ]:
test_tokens = tokenizer('Она знает язык животных и растений. ')

Токенайзер выдает для каждого токена его id из словаря, а также attention_mask:

In [ ]:
test_tokens

{'input_ids': [983, 558, 2419, 1167, 4217, 2829, 1074, 259, 279, 11188, 543, 260, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Чтобы посмотреть, на какие именно subword-токены токенайзер делит предложение, можно воспользоваться функцией tokenize:

In [ ]:
tokenizer.tokenize('Она знает язык животных и растений. ')

['▁О',
 'на',
 '▁зна',
 'ет',
 '▁язык',
 '▁живот',
 'ных',
 '▁',
 'и',
 '▁растени',
 'й',
 '.']

Attention mask означает, на какие токены нужно обратить внимание. Например, на padding tokens внимания обращать не нужно. Паддинг происходит, например, внутри батчей из множества предложений разных размеров, чтобы унифицировать их длины перед подачей в модель. Рассмотрим на примере двух предложений разной длины:

In [ ]:
padded_sequences = tokenizer(['Она знает язык животных и растений. ', 'Но не грибов'], padding=True)

Тут можно понять, что под номером 1 в словаре находится токен конца предложения, а под номером 0 - паддинг:

In [ ]:
padded_sequences["input_ids"]

[[983, 558, 2419, 1167, 4217, 2829, 1074, 259, 279, 11188, 543, 260, 1],
 [1894, 401, 259, 18195, 685, 1, 0, 0, 0, 0, 0, 0, 0]]

In [ ]:
padded_sequences["attention_mask"]

[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]]

Модель также выдает списки словарных id слов выходной последовательности. Чтобы превратить их в предложение, нужно воспользоваться декодированием:

In [ ]:
tokenizer.decode(padded_sequences["input_ids"][0])

'Она знает язык животных и растений.</s>'

In [ ]:
iterations = 15000

all_loss = 0

for i in range(iterations):
    xx, yy = get_batch(random.choice(lines))
    x = tokenizer(xx, return_tensors='pt', padding=True).to(model.device)
    y = tokenizer(yy, return_tensors='pt', padding=True).to(model.device)
    # если токен является паддингом, мы принудительно занижаем его значение, чтобы модель его не предсказывала
    y.input_ids[y.input_ids==0] = -100
    loss = model(
        input_ids=x.input_ids,
        attention_mask=x.attention_mask,
        labels=y.input_ids,
        decoder_attention_mask=y.attention_mask,
        # технический параметр, чтобы вернуть больше информации из модели. Может понадобиться для дебаггинга
        return_dict=True
    ).loss
    loss.backward()
    all_loss += loss.item()
    optimizer.step()
    optimizer.zero_grad()

    # progress report каждую тысячу эпох
    if (i > 0) and (i % 1000 == 0):
      print('Step: {0}, loss: {1}'.format(i, all_loss/i))

Step: 1000, loss: 0.3931904403483495
Step: 2000, loss: 0.3618276586169377
Step: 3000, loss: 0.3364092901402619
Step: 4000, loss: 0.32094595320126973
Step: 5000, loss: 0.30941244057128203
Step: 6000, loss: 0.3021479815442969
Step: 7000, loss: 0.2917734056395046
Step: 8000, loss: 0.2796247178956255
Step: 9000, loss: 0.2703631255485428
Step: 10000, loss: 0.2619859064316959
Step: 11000, loss: 0.2539618749556989
Step: 12000, loss: 0.2461185281163489
Step: 13000, loss: 0.23813675332373643
Step: 14000, loss: 0.2310902141031651


Теперь мы можем оценить работу нашей модели.

Во-первых, при помощи вызова model.eval() даем ей понять, что больше она не тренируется и веса менять не нужно.

Во-вторых, напишем уже знакомую вам по мастер-классу функцию для упрощения.

In [ ]:
model.eval()

def simplify(text, beams=3, grams=5, do_sample=True, num_return_sequences=1):
    x = tokenizer(text, return_tensors='pt', padding=True).to(model.device)
    # делаем так, чтобы максимальная длина генерируемого предложения не сильно превышала длину оригинального
    max_size = int(x.input_ids.shape[1] * 1.5 + 10)
    out = model.generate(**x, # генерируем упрощения для всех х в батче
                         encoder_no_repeat_ngram_size=grams, # все н-граммы такого размера, присутствовавшие в инпуте, не могут присутствовать в выдаче
                         num_beams=beams, # сколько "лучей" в beam_search (сколько последовательностей запоминаем)
                         max_length=max_size,
                         do_sample=do_sample, # если выставить False, будет greedy decoding
                         num_return_sequences=num_return_sequences # сколько возможных упрощений возвращать
                         )
    return [tokenizer.decode(o, skip_special_tokens=True) for o in out]

Скачаем метрику для оценки упрощения SARI и посмотрим, как наша модель упрощает предложения.

In [ ]:
! pip install git+https://github.com/feralvam/easse@master

  Cloning https://github.com/feralvam/easse (to revision master) to /tmp/pip-req-build-cbld64ub
  Running command git clone --filter=blob:none --quiet https://github.com/feralvam/easse /tmp/pip-req-build-cbld64ub
  Resolved https://github.com/feralvam/easse to commit 6a4352ec299ed03fda8ee45445ca43d9c7673e89
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/text-simplification-evaluation.git (to revision main) to /tmp/pip-install-wj2fmn4y/tseval_e57f6b32bae74984aadffc352cf77914
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/text-simplification-evaluation.git /tmp/pip-install-wj2fmn4y/tseval_e57f6b32bae74984aadffc352cf77914
  Resolved https://github.com/facebookresearch/text-simplification-evaluation.git to commit dea8863683ea5946fd50184883c9be7a7339e821
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ...

In [ ]:
from easse.sari import corpus_sari

Возьмем десять случайных предложений, которые наша модель еще не видела, упростим и оценим их.

In [ ]:
i = 0


while i < 10:
  row_id = random.randint(0, len(val_df)-1)
  current_orig = val_df['source'].iloc[row_id]
  current_out = simplify(current_orig)
  current_simple = val_df['target'].iloc[row_id]

  current_sari = corpus_sari(orig_sents=[current_orig], sys_sents=current_out, refs_sents=[[current_simple]])
  print('Original sentence: ', current_orig)
  print('Simplification: ', current_out)
  print('SARI: ', current_sari)
  print('*'*10)
  i += 1

Original sentence:  Все это и многое другое поставило русский народ перед тяжелыми испытаниями, многочисленными и сложными проблемами. 
Simplification:  ['Все эти и многие другие поставили русский народ перед сложными испытаниями.']
SARI:  8.226851851851851
**********
Original sentence:  В 1914 г. в составе Антанты (военного союза Великобритании, Франции и России против Германии) Россия вступила в Первую мировую войну, и Николай II занял пост верховного главнокомандующего. 
Simplification:  ['В 1914 г. во время войны Великобритания, Германия и Россия вступили в состав России.']
SARI:  6.852275839522652
**********
Original sentence:  Кориным (мозаика плафона станции «Комсомольская-кольцевая» Московского метро), И.С. 
Simplification:  ['Кориным.']
SARI:  2.0833333333333335
**********
Original sentence:  От глагола валять и происходит название этой распространённой в прошлом традиционной русской зимней обуви.
Simplification:  ['От глагола валяйте и происходит и название этой традиционной 